# distill-translate — 01: Base Training Loop (Part 1)

Fine-tune a **t5-small** student for **en-de** translation with **LoRA**, optionally **distilling** from a frozen **flan-t5-base** teacher. End-of-epoch checkpoints and all artifacts are persisted to **Google Drive** when running on Colab.

This notebook runs the full Part 1 flow on the **tiny** config:
1. Setup (install deps, mount Drive, route artifact paths)
2. SFT for 2 epochs with end-of-epoch checkpointing
3. Demonstrate **resume** (re-run picks up from the last completed epoch)
4. (Optional) **distillation** mode with a frozen teacher
5. **Evaluation** with sacreBLEU + saved predictions

> **Teacher-quality caveat (en-de).** t5-small / flan-t5-base are English-centric and are *not* strong en-de translators out of the box. Distillation quality is bounded by teacher quality, so here SFT is the primary quality driver and KD is wired up and validated as a mechanism. For real gains, fine-tune the teacher on en-de first.

**Colab verification gate 3 (acceptance):** run this end-to-end on a Colab **GPU** runtime (Runtime -> Change runtime type -> GPU) and confirm checkpoints land on Drive, resume works, and a BLEU score prints.

## 0. Clone the repo (Colab only)
On Colab, clone the project so the package is importable. Locally, skip this and run the notebook from the repo root.

In [ ]:
import os

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_URL = 'https://github.com/itsani-cmd/Distillation_cost_saver.git'
REPO_DIR = 'Distillation_cost_saver'

if IN_COLAB and not os.path.isdir(REPO_DIR):
    !git clone $REPO_URL

if IN_COLAB:
    os.chdir(REPO_DIR)
print('cwd:', os.getcwd())

## 1. Setup: install deps, mount Drive, resolve paths
`setup_colab()` installs `requirements.txt`, mounts Drive, and returns the Drive-routed artifact paths. Off-Colab it just returns local paths, so this notebook is runnable locally too.

In [ ]:
from utils.colab_utils import setup_colab

RUN_NAME = 'tiny'
paths = setup_colab(run_name=RUN_NAME)
print('checkpoints ->', paths.checkpoints)
print('logs        ->', paths.logs)
print('outputs     ->', paths.outputs)

## 2. Load the tiny config
Small subset + 2 epochs so the whole loop runs quickly. On a GPU, set `training.mixed_precision` to `fp16` in `configs/tiny.yaml` for speed.

In [ ]:
from src.config import load_config

cfg = load_config('configs/tiny.yaml')
print('student :', cfg.model.student_name_or_path)
print('teacher :', cfg.model.teacher_name_or_path)
print('epochs  :', cfg.training.num_epochs)
print('kd_weight:', cfg.distillation.kd_weight, '(0 => pure SFT)')

## 3. SFT training (2 epochs) with end-of-epoch checkpoints
Artifacts (LoRA adapter + optimizer/scheduler/RNG state) are written to Drive at the end of each epoch.

In [ ]:
from src.train import Trainer

trainer = Trainer(cfg, paths=paths)
result = trainer.train()
print('SFT done:', result)

In [ ]:
# Confirm checkpoints exist on Drive.
print(sorted(p.name for p in paths.checkpoints.glob('epoch_*')))

## 4. Demonstrate resume
A fresh `Trainer` on the same run finds the last completed epoch and resumes from the next one. Since both epochs are already done, `start_epoch` should equal `num_epochs` (nothing left to train), proving resume state was restored.

In [ ]:
resumed = Trainer(cfg, paths=paths)
resumed.maybe_resume()
print('resumed start_epoch:', resumed.start_epoch, '(== num_epochs means fully trained)')
print('restored global_step:', resumed.global_step)

## 5. (Optional) Distillation mode
Point the config at a teacher and give KD some weight, then train a fresh run. The teacher is loaded frozen and only used for forward passes. Per-step logs show `ce`, `kd`, and the combined `loss`.

> Uses `google/flan-t5-small` as a lightweight teacher for the demo. Swap in `google/flan-t5-base` on a GPU. Remember the en-de teacher-quality caveat above.

In [ ]:
from src.config import load_config
from src.train import Trainer
from utils.colab_utils import get_project_paths

kd_cfg = load_config('configs/tiny.yaml')
kd_cfg.run_name = 'tiny_kd'
kd_cfg.model.teacher_name_or_path = 'google/flan-t5-small'
kd_cfg.distillation.kd_weight = 0.5
kd_cfg.training.num_epochs = 1
print('distillation_enabled:', kd_cfg.distillation_enabled)

kd_paths = get_project_paths(kd_cfg.run_name)
kd_trainer = Trainer(kd_cfg, paths=kd_paths)
print('teacher frozen:', all(not p.requires_grad for p in kd_trainer.teacher.parameters()))
kd_result = kd_trainer.train()
print('KD done:', kd_result)

## 6. Evaluation + predictions
`train()` already runs a final sacreBLEU evaluation and writes predictions. You can also re-run evaluation standalone on the trained student.

In [ ]:
from src.evaluate import evaluate_model

pred_path = paths.outputs / 'tiny_eval_rerun.jsonl'
eval_out = evaluate_model(
    trainer.accelerator.unwrap_model(trainer.model),
    trainer.eval_loader,
    trainer.tokenizer,
    cfg,
    device=trainer.accelerator.device,
    output_path=pred_path,
)
print('BLEU:', round(eval_out['bleu'], 2))
print('predictions saved to:', pred_path)
for s, r, h in list(zip(eval_out['sources'], eval_out['references'], eval_out['hypotheses']))[:3]:
    print('\nSRC:', s)
    print('REF:', r)
    print('HYP:', h)

## Done — Part 1 acceptance
You fine-tuned a small Transformer with LoRA on en-de (SFT and distillation), resumed training, and produced a BLEU score — all artifacts persisted to Drive.

Part 2 (per-layer observation) and Part 3 (early-exit / layer-skipping) build on the hook points left in `src/train.py::Trainer._train_step` and the plain HF model in `src/model.py`.